# MRC Combiner — Full-Path Analysis

`mrc_combiner.v` is RX path stage 9 — the serialised int8 MAC that forms
$\hat{y}[n] = \sum_k W_k \cdot x_k[n]$, `>>>1` guard, optional
`COMB_POST_GAIN_SHIFT`, saturate to int8 (see the module header for the
Option-A 8x8-multiplier-shared, 11-state pipeline).

This is the block's end-to-end walkthrough notebook. Two earlier, narrower
notebooks already cover pieces of this block in more depth and are not
re-derived here, only summarised and tied together:

- **`07_mrc_weight_quantisation.ipynb`** — does int8 *weight* quantisation
  (`W_MAX` budget) hurt combining SNR, as a function of inter-branch SNR
  spread? (idealised weights, no amplitude/clipping model)
- **`08_mrc_output_headroom.ipynb`** — does the raw 8-bit combiner
  arithmetic *clip* before the re-modulator, for representative branch
  amplitudes and weight scales? (synthetic weight values, no weight-gen path)

Sections 1-3 here reproduce each notebook's model at Trouper's realistic
operating point (branch amplitude ~90 counts on the int8 ADC scale,
`W_MAX=45`) as a sanity check, and Section 4 adds a full-pipeline analysis
neither prior notebook does: real weights from `weight_generation.py`'s
`WeightGenerator` HW FSM model (SHIFT->CALIBRATE->COMPUTE->SCALE, matching
`weight_gen.v`) fed straight into the bit-exact `mrc_combiner.v` model, in
noise, comparing MRC/EGC/SC combining modes.

**Known architecture caveat** (see `CLAUDE.md`, System Architecture and
`sim/models/training_accumulator.py` docstring): `trouper_top.v` does **not**
instantiate `weight_gen.v` — production weights come from PicoRV32 firmware's
eigenvector path (`compute_eigvec_fw`) reading `Zpair_kl`/`Zdiag_k` from the
register bank. `weight_gen.v`'s HW MRC/EGC/SC modes are exercised here as a
documented alternate weight source with a fully-modeled hardware datapath —
not a claim that this is the mode driving silicon today.

In [1]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path('../..').resolve()))

import numpy as np
import matplotlib.pyplot as plt

from sim.models.weight_generation import WeightGenerator
from sim.models.receiver import nonfft_combine_rtl_int8w

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

RNG = np.random.default_rng(0)
NR = 4
PLOT_DIR = pathlib.Path('../plots')
PLOT_DIR.mkdir(parents=True, exist_ok=True)


---
## 1  Noiseless correctness

Build a 4-branch scenario with known channel `h`, quantise weights via the
`weight_gen.v` HW MRC mode (`WeightGenerator`), combine int8-quantised
samples through `nonfft_combine_rtl_int8w()` (the bit-exact model of
`mrc_combiner.v`'s `net_rshift = 8 - post_gain_shift` datapath — see the RTL
header, commit that replaced the old two-step `(acc >>> 8) << pgs`), and
check the result correlates with the phase/amplitude of an ideal
`conj(h)`-weighted combine.

In [2]:
gamma_db = [0, -3, -6, -9]                 # branch SNR profile
gamma = 10.0 ** (np.array(gamma_db) / 10.0)
phases = RNG.uniform(0, 2 * np.pi, NR)
h = np.sqrt(gamma) * np.exp(1j * phases)

n_samples = 400
s = np.exp(1j * RNG.uniform(0, 2 * np.pi, n_samples))       # unit-power symbol stream
amp = 90.0
x = amp * h[:, None] * s[None, :]
x_i8 = np.clip(np.round(x.real), -127, 127) + 1j * np.clip(np.round(x.imag), -127, 127)

# weight_gen.v HW MRC path: SHIFT (sf=7) -> CALIBRATE (none) -> COMPUTE -> SCALE
Z_j = h * amp**2 * n_samples / NR      # rough training-accumulator-scale proxy
w_mrc, sf_shift = WeightGenerator(mode='mrc').process(Z_j, sf=7)

y_rtl = nonfft_combine_rtl_int8w(x_i8, w_mrc, post_gain_shift=0)
y_ideal = np.sum(np.conj(h)[:, None] * x_i8, axis=0)

corr = np.abs(np.vdot(y_rtl, y_ideal)) / (np.linalg.norm(y_rtl) * np.linalg.norm(y_ideal))
print(f"weight_gen.v SHIFT amount: sf={sf_shift}")
print(f"HW MRC weights: {np.round(w_mrc, 5)}")
print(f"normalised correlation, RTL combiner output vs ideal conj(h)-MRC: {corr:.6f}")
assert corr > 0.999, "bit-exact combiner output should track the ideal MRC phase/amplitude closely"

print()
for mode in ['mrc', 'egc', 'sc', 'bypass']:
    w, _ = WeightGenerator(mode=mode).process(Z_j, sf=7)
    print(f"{mode:7s} weights: {np.round(w, 4)}")


weight_gen.v SHIFT amount: sf=7
HW MRC weights: [-0.03152+0.03662j -0.00427-0.03394j  0.02341-0.0062j   0.01703-0.0018j ]
normalised correlation, RTL combiner output vs ideal conj(h)-MRC: 0.999972

mrc     weights: [-0.0315+0.0366j -0.0043-0.0339j  0.0234-0.0062j  0.017 -0.0018j]
egc     weights: [-0.652 +0.7582j -0.1241-0.9923j  0.9671-0.2545j  0.9947-0.1033j]
sc      weights: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
bypass  weights: [1.+0.j 0.+0.j 0.+0.j 0.+0.j]


`sc` correctly selects antenna 0 (the strongest branch in `gamma_db`, unit
weight); `bypass` does the same by construction (first enabled antenna,
matching `mrc_combiner.v`'s `bypass_ant` mux which is independent of channel
quality — firmware picks the branch, not the FSM). `egc` produces unit-
magnitude phase-only weights on all four branches. `mrc` weights are tiny in
this Q1.15-normalised representation because `weight_gen.v`'s SHIFT/COMPUTE
path scales by a peak-derived power-of-two, not by full range — this is
exactly what `nonfft_combine_rtl_int8w()`'s internal peak-to-`120` rescale
compensates for before the weights hit the RTL's 8-bit multiplier input.

---
## 2  Weight quantisation loss (operating-point check)

Full sweep is `07_mrc_weight_quantisation.ipynb`. Here: just the numbers at
Trouper's actual `W_MAX=45` design point, at SNR spreads representative of a
4-branch Rayleigh-faded link (`0`, `6`, `12`, `18` dB strongest-to-weakest).

In [3]:
def combining_snr(w, h):
    sig = np.abs(np.dot(w, h)) ** 2
    nse = float(np.sum(np.abs(w) ** 2))
    return sig / nse if nse > 0 else 0.0


def quantise_weights(w_float, w_max):
    peak = np.max(np.abs(w_float))
    if peak == 0:
        return np.zeros_like(w_float, dtype=complex)
    w_scaled = w_float * (w_max / peak)
    return np.round(w_scaled.real) + 1j * np.round(w_scaled.imag)


W_MAX = 45
N_TRIALS = 5_000
spreads_dB = [0.0, 6.0, 12.0, 18.0]

print(f"{'spread (dB)':>12}  {'mean loss (dB)':>16}  {'p99 loss (dB)':>16}")
print('-' * 48)
for spread_dB in spreads_dB:
    gamma_k = 10.0 ** (-np.linspace(0.0, spread_dB, NR) / 10.0)
    snr_ideal = float(np.sum(gamma_k))
    phases_mc = RNG.uniform(0, 2 * np.pi, (N_TRIALS, NR))
    h_mat = np.sqrt(gamma_k) * np.exp(1j * phases_mc)
    w_opt = np.conj(h_mat)
    peaks = np.max(np.abs(w_opt), axis=1, keepdims=True)
    w_sc = w_opt * (W_MAX / np.where(peaks > 0, peaks, 1.0))
    w_q = np.round(w_sc.real) + 1j * np.round(w_sc.imag)
    sig_q = np.abs(np.sum(w_q * h_mat, axis=1)) ** 2
    nse_q = np.sum(np.abs(w_q) ** 2, axis=1)
    snr_q = np.where(nse_q > 0, sig_q / nse_q, 0.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        loss_dB = 10.0 * np.log10(np.where(snr_q > 0, snr_ideal / snr_q, np.nan))
    print(f"{spread_dB:>12.1f}  {np.nanmean(loss_dB):>16.5f}  {np.nanpercentile(loss_dB, 99):>16.5f}")


 spread (dB)    mean loss (dB)     p99 loss (dB)
------------------------------------------------
         0.0           0.00025           0.00054
         6.0           0.00049           0.00098
        12.0           0.00063           0.00131
        18.0           0.00085           0.00167


Matches `07`'s finding: at `W_MAX=45`, quantisation loss stays below
`0.002 dB` even at `18 dB` spread. int8 weight resolution is not the limiting
factor for Trouper's NR=4 MRC — see Section 3.

---
## 3  Output headroom / clipping (operating-point check)

`08_mrc_output_headroom.ipynb`'s headline finding is that `A=90, W=45`
(Trouper's nominal operating point) clips the raw combiner sum. Reproducing
that exact case against the model used in Section 1 above **does not**
reproduce that conclusion — and the reason is itself worth recording.

`08`'s `rtl_complex_mac()` shifts with the *old* two-step scheme
(`acc >>> 1` then `<< post_gain_shift`), which `mrc_combiner.v`'s own header
comment says was superseded: *"Output shift: acc >>> (8 − pgs) — single
combined shift replacing the old two-step (acc >>> 8) << pgs. Eliminates
amplified truncation."* `nonfft_combine_rtl_int8w()` (used in Section 1) and
the `rtl_complex_mac()` below both implement the **current** combined-shift
RTL; `08`'s helper of the same name does not. So `08`'s numeric headroom
conclusions predate that shift refactor and should be read as historical,
not current-RTL, findings.

In [4]:
def arshift(val, shift):
    return val >> shift


def rtl_complex_mac(x, w, post_gain_shift=0, remod_backoff_shift=1):
    x_i, x_q = np.real(x).astype(int), np.imag(x).astype(int)
    w_i, w_q = np.real(w).astype(int), np.imag(w).astype(int)
    acc_i = int(np.sum(w_i * x_i - w_q * x_q))
    acc_q = int(np.sum(w_i * x_q + w_q * x_i))
    net_rshift = 8 - post_gain_shift
    comb_i = int(np.clip(arshift(acc_i, net_rshift), -128, 127))
    comb_q = int(np.clip(arshift(acc_q, net_rshift), -128, 127))
    remod_i = arshift(comb_i, remod_backoff_shift)
    remod_q = arshift(comb_q, remod_backoff_shift)
    clipped = (arshift(acc_i, net_rshift) != comb_i) or (arshift(acc_q, net_rshift) != comb_q)
    return comb_i, comb_q, remod_i, remod_q, clipped


for A, W in [(90, 45), (90, 1), (32, 1), (16, 3)]:
    x = np.array([A + 0j] * NR)
    w = np.array([W + 0j] * NR)
    comb_i, comb_q, remod_i, remod_q, clipped = rtl_complex_mac(x, w)
    print(f"A={A:3d}  W={W:3d}  comb_y={comb_i:5d}  remod_in={remod_i:5d}  clipped={clipped}")


A= 90  W= 45  comb_y=   63  remod_in=   31  clipped=False
A= 90  W=  1  comb_y=    1  remod_in=    0  clipped=False
A= 32  W=  1  comb_y=    0  remod_in=    0  clipped=False
A= 16  W=  3  comb_y=    0  remod_in=    0  clipped=False


With the current combined-shift RTL, `A=90, W=45` does **not** clip
(`comb_y=63`, well inside `±127`) — the opposite of `08`'s conclusion for
the same nominal operating point, because `08` modeled the superseded
two-step shift. `08` should be re-run against `nonfft_combine_rtl_int8w()`
(or an updated local `rtl_complex_mac()` using `net_rshift = 8 -
post_gain_shift`) before its headroom conclusions are relied on. The
`A=90, W=1` / `A=32, W=1` / `A=16, W=3` cases stay far from saturation
either way — the corrected picture is that fixed small integer weights have
generous headroom under the current combiner; Section 4 shows the real risk
comes from a different source: how large a weight magnitude the weight
generation path (or firmware policy) actually produces.

---
## 4  Full pipeline: HW weight-gen modes through the bit-exact combiner, in noise

Neither `07` nor `08` runs real `weight_gen.v`-path weights (Section 1's
`WeightGenerator`) through the RTL combiner model under AWGN. This section
does, comparing the three HW combining modes (`mrc`, `egc`, `sc`) at a
clip-free amplitude, then separately shows how the same pipeline clips at
the realistic `90`-count amplitude from Section 3 when `nonfft_combine_rtl_int8w()`'s
weight-scaling convention (peak branch weight normalised to `~120`, not `45`)
is used — the actual source of the headroom risk, now that Section 3 has
ruled out the shift arithmetic itself as the cause.

In [5]:
def run_case(spread_db, amp, n_samples, snr_db, mode, seed, post_gain_shift=0):
    rng = np.random.default_rng(seed)
    gamma_db = -np.linspace(0.0, spread_db, NR)
    gamma = 10.0 ** (gamma_db / 10.0)
    phases = rng.uniform(0, 2 * np.pi, NR)
    h = np.sqrt(gamma) * np.exp(1j * phases)

    s = np.exp(1j * rng.uniform(0, 2 * np.pi, n_samples))
    clean = amp * h[:, None] * s[None, :]
    noise_amp = amp / np.sqrt(10 ** (snr_db / 10))
    noise = noise_amp / np.sqrt(2) * (rng.standard_normal(clean.shape) + 1j * rng.standard_normal(clean.shape))
    x = clean + noise
    x_i8 = np.clip(np.round(x.real), -127, 127) + 1j * np.clip(np.round(x.imag), -127, 127)

    Z_j = h * amp**2 * n_samples / NR
    w, _ = WeightGenerator(mode=mode).process(Z_j, sf=7)
    y = nonfft_combine_rtl_int8w(x_i8, w, post_gain_shift=post_gain_shift)

    clip_frac = float(np.mean((np.abs(y.real) >= 127) | (np.abs(y.imag) >= 127)))
    proj = np.vdot(s, y) / n_samples
    sig_est = proj * s
    resid = y - sig_est
    snr_out = 10 * np.log10(np.sum(np.abs(sig_est) ** 2) / np.sum(np.abs(resid) ** 2))
    return clip_frac, snr_out


spreads_dB = np.array([0, 3, 6, 9, 12, 15, 18], dtype=float)
N_SEEDS = 12
INPUT_SNR_DB = 5.0

# Clip-free amplitude (headroom set aside) — isolates the mode-vs-spread comparison
snr_by_mode = {}
for mode in ['mrc', 'egc', 'sc']:
    means = []
    for spread in spreads_dB:
        vals = [run_case(spread, amp=20.0, n_samples=4000, snr_db=INPUT_SNR_DB, mode=mode, seed=seed)[1]
                for seed in range(N_SEEDS)]
        means.append(np.mean(vals))
    snr_by_mode[mode] = np.array(means)
    print(f"{mode}: " + " ".join(f"{v:5.2f}" for v in means))

# Realistic amplitude (90 counts, Section 3's operating point) — same sweep, mode=mrc only
clip_by_spread = [np.mean([run_case(spread, amp=90.0, n_samples=4000, snr_db=INPUT_SNR_DB, mode='mrc', seed=seed)[0]
                            for seed in range(N_SEEDS)])
                   for spread in spreads_dB]
print()
print("mrc clip_frac @ 90-count amplitude:", [f"{c:.2f}" for c in clip_by_spread])


mrc: 10.98  9.62  8.53  7.68  7.04  6.55  6.18
egc: 10.98  9.56  8.27  7.11  6.10  5.20  4.40
sc:  4.90  4.94  4.94  4.94  4.94  4.94  4.94



mrc clip_frac @ 90-count amplitude: ['0.80', '0.44', '0.18', '0.07', '0.03', '0.01', '0.00']


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
colours = {'mrc': '#1f77b4', 'egc': '#ff7f0e', 'sc': '#2ca02c'}
for mode, col in colours.items():
    ax.plot(spreads_dB, snr_by_mode[mode], 'o-', color=col, lw=2, label=mode.upper())
ax.axhline(INPUT_SNR_DB, color='gray', ls=':', lw=1, label=f'single-branch SNR ({INPUT_SNR_DB:.0f} dB)')
ax.set_xlabel('Inter-branch SNR spread (dB)')
ax.set_ylabel('Combined output SNR (dB)')
ax.set_title('HW combining modes through bit-exact RTL model\n(amp=20, clip-free)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(spreads_dB, clip_by_spread, 'o-', color='#d62728', lw=2)
ax.set_xlabel('Inter-branch SNR spread (dB)')
ax.set_ylabel('Fraction of output samples clipped')
ax.set_title('MRC mode at realistic amp=90\n(Section 3 operating point)')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

fig.tight_layout()
plt.savefig(PLOT_DIR / 'mrc_full_pipeline_modes_and_clipping.png', bbox_inches='tight')
plt.show()
print('Saved: sim/plots/mrc_full_pipeline_modes_and_clipping.png')


Saved: sim/plots/mrc_full_pipeline_modes_and_clipping.png


---
## Summary

| Question | Finding |
|---|---|
| Noiseless correctness (Section 1) | `weight_gen.v` HW-mode weights, run through the bit-exact `mrc_combiner.v` model, correlate `>0.999` with an ideal `conj(h)`-weighted combine; `sc`/`bypass`/`egc` modes behave as specified |
| Weight quantisation (Section 2, full detail in `07`) | Not the limiting factor — `W_MAX=45` loses `<0.002 dB` even at `18 dB` branch-SNR spread |
| **Output headroom, corrected (Section 3)** | **Corrects `08`.** `08_mrc_output_headroom.ipynb` models the pre-refactor two-step combiner shift (`>>>1` then `<<pgs`), which `mrc_combiner.v`'s header says was replaced by a single combined shift `>>>(8-pgs)`. Under the *current* RTL, `A=90, W=45` does **not** clip (`comb_y=63` vs the `±127` bound) — `08` should be re-run against the current-shift model before its conclusions are used for firmware policy |
| **Mode comparison, clip-free (Section 4)** | **New in this notebook.** MRC $\ge$ EGC $\ge$ SC in combined SNR, and the MRC-over-EGC gap widens with SNR spread (`~0.1 dB` at `0 dB` spread -> `~1.8 dB` at `18 dB` spread in this sweep); SC output SNR is flat at roughly the strongest-branch input SNR regardless of spread, as expected for a selection combiner |
| **Full-pipeline clipping vs spread (Section 4)** | **New in this notebook.** At the realistic `90`-count amplitude, clipping *is* real once weights use `nonfft_combine_rtl_int8w()`'s peak-to-`120` convention (much larger than the `W_MAX=45` figure in `07`/`08`): clip fraction runs `0.80` at `0 dB` spread down to `~0` by `12 dB` spread, since a large spread naturally suppresses the weaker branches' contribution to the coherent sum |

**Decision-useful implication:** the actual headroom risk is not the
combiner's shift arithmetic (Section 3 shows current RTL has more margin
than `08` reported) — it's whatever peak weight magnitude the upstream
weight-generation policy (HW `weight_gen.v` FSM, or firmware's eigenvector
path) actually produces relative to branch amplitude. `07`'s int8
*precision* margin is real but orthogonal; a firmware/HW weight-scaling rule
still needs to be derived from the combiner's `±127` bound at whatever peak
weight the live path emits, not assumed from a fixed `W_MAX` figure.
`08_mrc_output_headroom.ipynb` should be updated to the current combined-
shift model so its conclusions and this notebook's don't disagree.